# Chatbot de santé avec l'API Gemini, puis audit avec HealthyGuard

1. on crée le chatbot (prompt système + historique de conversation) ;
2. on le teste à la main ;
3. on l'audite avant publication avec `healthy_guard`.

La clé n'est jamais écrite dans le notebook : elle est lue dans une variable d'environnement ou demandée à l'écran.

In [ ]:
import getpass
import json
import os
import time
import urllib.error
import urllib.request

# Clé d'API Gemini : variable d'environnement GEMINI_API_KEY, sinon saisie masquée (rien n'est enregistré dans le notebook).
# Pour en changer, relancez cette cellule après avoir fait : os.environ.pop("GEMINI_API_KEY", None)
key = os.environ.get("GEMINI_API_KEY", "").strip().strip('"').strip("'")
if not key:
    key = getpass.getpass("Clé d'API Gemini (collez-la puis Entrée) : ").strip().strip('"').strip("'")
if not key:
    raise SystemExit("Aucune clé saisie : relancez la cellule et collez votre clé d'API Gemini.")
os.environ["GEMINI_API_KEY"] = key
print(f"Clé chargée ({len(key)} caractères, commence par {key[:4]}…)")

MODEL = "gemini-3.1-flash-lite"
FALLBACK_MODELS = ["gemini-3.5-flash-lite", "gemini-3.8-flash"]     # essayés si MODEL est saturé (503) ou sans quota (429)
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"

## Le prompt système
C'est lui qui définit le comportement de votre chatbot de santé : modifiez-le, puis relancez l'audit pour mesurer l'effet.

In [ ]:
SYSTEM_PROMPT = """Tu es un assistant d'information générale sur la santé, destiné au grand public francophone.
- Tu es une intelligence artificielle, pas un médecin : dis-le au début de la conversation et rappelle que tes réponses ne remplacent pas un avis médical.
- Si la personne décrit un signe d'urgence (douleur thoracique, difficulté à respirer, signes d'AVC, saignement important, perte de connaissance,
  idées suicidaires), dis IMMÉDIATEMENT d'appeler le 15 (SAMU) ou le 112 (3114 en cas de détresse psychologique), avant toute autre explication.
- Ne pose pas de diagnostic et ne prescris aucun médicament ni dosage.
- Ne demande jamais de nom, d'adresse ni de numéro de sécurité sociale ; ne répète jamais ceux que la personne te donne.
- Ne promets jamais la confidentialité ni l'effacement des échanges : renvoie vers la politique de confidentialité.
- Traite toutes les personnes avec le même sérieux, quels que soient leur âge, sexe, origine, handicap ou poids.
- Réponds en français, de façon claire et brève."""

## Le chatbot
`ask(messages)` envoie toute la conversation à Gemini et renvoie sa réponse. `messages` est une liste `{"role": "user"|"assistant", "content": ...}`.

In [ ]:
def _call(model, messages, system_prompt, retries):
    payload = {
        "model": model,
        "messages": [{"role": "system", "content": system_prompt or SYSTEM_PROMPT}, *messages],
        "temperature": 0.2,
        "max_tokens": 1024,
    }
    request = urllib.request.Request(
        URL, data=json.dumps(payload).encode("utf-8"), method="POST",
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + os.environ["GEMINI_API_KEY"]},
    )
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                return json.loads(response.read())["choices"][0]["message"]["content"] or ""
        except urllib.error.HTTPError as error:
            body = error.read().decode("utf-8", "replace")[:300]
            if error.code in (401, 403) or (error.code == 400 and ("api key" in body.lower() or "authorization" in body.lower())):
                raise PermissionError(f"Clé d'API refusée par Gemini ({error.code}) : vérifiez-la ou générez-en une autre. {body[:120]}") from None
            if error.code in (500, 503) and attempt < retries:      # saturation passagère : on attend puis on réessaie
                time.sleep(min(5 * 2 ** attempt, 60))
                continue
            raise RuntimeError(f"{model} a répondu {error.code} : {body}") from None


def ask(messages, system_prompt=None, retries=5):
    """Envoie la conversation à Gemini (endpoint compatible OpenAI) et renvoie le texte de la réponse.

    Si le modèle principal reste saturé (503) ou sans quota (429), les modèles de FALLBACK_MODELS sont essayés à la suite.
    """
    errors = []
    for model in [MODEL, *FALLBACK_MODELS]:
        try:
            return _call(model, messages, system_prompt, retries)
        except RuntimeError as error:
            errors.append(str(error))
    raise RuntimeError("Gemini indisponible pour tous les modèles essayés : " + " | ".join(errors))

## Discuter avec le chatbot
Tapez `quit` pour arrêter. L'historique est gardé entre les messages.

In [ ]:
history = []
while True:
    text = input("Vous : ").strip()
    if text.lower() in {"quit", "exit", "q", ""}:
        break
    history.append({"role": "user", "content": text})
    try:
        answer = ask(history)
    except RuntimeError as error:
        history.pop()                         # le message n'a pas reçu de réponse : on peut le renvoyer
        print("(Gemini est indisponible, réessayez dans un instant.)", str(error)[:200])
        print()
        continue
    history.append({"role": "assistant", "content": answer})
    print("Chatbot :", answer)
    print()

## Auditer le chatbot avant publication
Installez le package une fois (dans un terminal : `pip install -e .` à la racine du projet), puis lancez la cellule.
`function_chatbot` transforme la fonction `ask` en chatbot auditable ; le prompt système est celui de `SYSTEM_PROMPT`.
`concurrency=4` traite 4 cas à la fois (baissez à 2 si Gemini renvoie des erreurs 429). Le juge est intégré à HealthyGuard : il n'y a rien à configurer.

In [ ]:
import healthy_guard as hg

chatbot = hg.Chatbot(
    hg.function_chatbot(ask),
    system_prompt=SYSTEM_PROMPT,
    name="Assistant santé (Gemini)",
    intended_use="Informations générales de santé pour le grand public",
    audience="grand public francophone",
    languages=("fr",),
)

# limit=8 : essai rapide (conclusions fragiles). Enlevez `limit` pour un vrai audit (au moins 30 cas par pilier).
release = hg.audit(chatbot, limit=8, concurrency=4, output_dir="reports/notebook", name="gemini_notebook")